# 性状分离模拟器：WwYy 自交

本 notebook 模拟 `WwYy x WwYy` 的自交过程。

- `W/w` 控制胚乳性状：`W_` 为非糯性，`ww` 为糯性。
- `Y/y` 控制粒形：`Y_` 为长粒，`yy` 为短粒。
- 理论性状组合比例：非糯性长粒 : 非糯性短粒 : 糯性长粒 : 糯性短粒 = `9 : 3 : 3 : 1`。

模拟器支持两种方式：

1. 随机连续实验：自动随机抽取两个亲本的配子，连续模拟多次。
2. 人工指定配子：手动输入两个亲本配子，例如 `WY` 和 `wy`，查看该次受精结果。

In [2]:
import random
from collections import Counter
from html import escape
from itertools import product

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None
    def display(obj):
        print(obj)

try:
    import pandas as pd
except ImportError:
    pd = None

if plt is not None:
    plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS", "Noto Sans CJK SC", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False

LOCUS_CONFIG = [
    {
        "gene": "W",
        "dominant": "W",
        "recessive": "w",
        "trait_name": "胚乳性状",
        "dominant_trait": "非糯性",
        "recessive_trait": "糯性",
    },
    {
        "gene": "Y",
        "dominant": "Y",
        "recessive": "y",
        "trait_name": "粒形",
        "dominant_trait": "长粒",
        "recessive_trait": "短粒",
    },
]

PARENT_GENOTYPE = "WwYy"
PHENOTYPE_ORDER = ["非糯性长粒", "非糯性短粒", "糯性长粒", "糯性短粒"]
THEORETICAL_PHENOTYPE_RATIO = {
    "非糯性长粒": 9 / 16,
    "非糯性短粒": 3 / 16,
    "糯性长粒": 3 / 16,
    "糯性短粒": 1 / 16,
}


def show_table(rows):
    """在 notebook 中优先用 pandas 表格展示；没有 pandas 时退回普通打印。"""
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        for row in rows:
            print(row)


def parse_genotype(genotype, loci=LOCUS_CONFIG):
    if len(genotype) != 2 * len(loci):
        raise ValueError(f"基因型长度应为 {2 * len(loci)}，例如 WwYy。")

    pairs = []
    for index, locus in enumerate(loci):
        pair = list(genotype[2 * index : 2 * index + 2])
        allowed = {locus["dominant"], locus["recessive"]}
        if any(allele not in allowed for allele in pair):
            raise ValueError(f"第 {index + 1} 对基因应只包含 {allowed}，实际为 {pair}。")
        pairs.append(pair)
    return pairs


def normalize_pair(alleles, dominant, recessive):
    order = {dominant: 0, recessive: 1}
    return sorted(alleles, key=lambda allele: order[allele])


def possible_gametes(parent_genotype=PARENT_GENOTYPE):
    """列出亲本可能产生的配子及理论概率。"""
    pairs = parse_genotype(parent_genotype)
    all_gametes = ["".join(items) for items in product(*pairs)]
    counts = Counter(all_gametes)
    total = sum(counts.values())
    return {gamete: counts[gamete] / total for gamete in sorted(counts)}


def make_random_gamete(parent_genotype=PARENT_GENOTYPE, rng=None):
    """模拟一个亲本经过减数分裂随机产生一个配子。"""
    rng = rng or random
    pairs = parse_genotype(parent_genotype)
    return "".join(rng.choice(pair) for pair in pairs)


def make_offspring_from_gametes(gamete1, gamete2, loci=LOCUS_CONFIG):
    """人工指定两个配子，返回后代基因型和两个性状组合。"""
    if len(gamete1) != len(loci) or len(gamete2) != len(loci):
        raise ValueError(f"每个配子应包含 {len(loci)} 个基因，例如 WY、Wy、wY、wy。")

    genotype_parts = []
    trait_parts = []
    row = {"亲本甲配子": gamete1, "亲本乙配子": gamete2}

    for index, locus in enumerate(loci):
        allowed = {locus["dominant"], locus["recessive"]}
        alleles = [gamete1[index], gamete2[index]]
        if any(allele not in allowed for allele in alleles):
            raise ValueError(f"第 {index + 1} 对基因只允许 {allowed}，实际为 {alleles}。")

        pair = normalize_pair(alleles, locus["dominant"], locus["recessive"])
        genotype_parts.append("".join(pair))

        if pair == [locus["recessive"], locus["recessive"]]:
            trait = locus["recessive_trait"]
        else:
            trait = locus["dominant_trait"]
        trait_parts.append(trait)
        row[locus["trait_name"]] = trait

    row["后代基因型"] = "".join(genotype_parts)
    row["性状组合"] = "".join(trait_parts)
    return row


def run_random_experiments(n=50, parent1=PARENT_GENOTYPE, parent2=PARENT_GENOTYPE, seed=None):
    """连续进行 n 次随机抽取配子的模拟实验。"""
    rng = random.Random(seed)
    records = []
    for experiment_id in range(1, n + 1):
        gamete1 = make_random_gamete(parent1, rng)
        gamete2 = make_random_gamete(parent2, rng)
        row = make_offspring_from_gametes(gamete1, gamete2)
        row = {"实验序号": experiment_id, **row}
        records.append(row)
    return records


def summarize_results(records):
    total = len(records)
    phenotype_counts = Counter(row["性状组合"] for row in records)
    genotype_counts = Counter(row["后代基因型"] for row in records)

    phenotype_rows = []
    for phenotype in PHENOTYPE_ORDER:
        count = phenotype_counts[phenotype]
        phenotype_rows.append(
            {
                "性状组合": phenotype,
                "数量": count,
                "实际比例": count / total if total else 0,
                "理论比例": THEORETICAL_PHENOTYPE_RATIO[phenotype],
            }
        )

    genotype_rows = [
        {"后代基因型": genotype, "数量": count, "实际比例": count / total if total else 0}
        for genotype, count in sorted(genotype_counts.items())
    ]
    return phenotype_rows, genotype_rows


def plot_results_svg(records):
    """无 matplotlib 时，用 Jupyter 原生 HTML/SVG 画实际次数与理论次数对比图。"""
    total = len(records)
    phenotype_counts = Counter(row["性状组合"] for row in records)
    actual_counts = [phenotype_counts[item] for item in PHENOTYPE_ORDER]
    theoretical_counts = [THEORETICAL_PHENOTYPE_RATIO[item] * total for item in PHENOTYPE_ORDER]
    max_count = max(actual_counts + theoretical_counts + [1])

    width = 860
    height = 430
    left = 70
    bottom = 330
    chart_height = 240
    group_width = 170
    bar_width = 34

    parts = [
        f'<svg width="{width}" height="{height}" viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{width / 2}" y="32" text-anchor="middle" font-size="20" font-weight="700">性状组合统计（n={total}）</text>',
        f'<line x1="{left}" y1="{bottom}" x2="{width - 60}" y2="{bottom}" stroke="#222"/>',
        f'<line x1="{left}" y1="{bottom}" x2="{left}" y2="{bottom - chart_height}" stroke="#222"/>',
        '<rect x="640" y="56" width="18" height="18" fill="#3b82f6"/>',
        '<text x="666" y="70" font-size="14">实际次数</text>',
        '<rect x="740" y="56" width="18" height="18" fill="#f59e0b"/>',
        '<text x="766" y="70" font-size="14">理论期望次数</text>',
    ]

    for tick in range(0, 5):
        value = max_count * tick / 4
        y = bottom - chart_height * tick / 4
        parts.append(f'<line x1="{left - 4}" y1="{y}" x2="{width - 60}" y2="{y}" stroke="#e5e7eb"/>')
        parts.append(f'<text x="{left - 10}" y="{y + 5}" text-anchor="end" font-size="12">{value:.0f}</text>')

    for index, phenotype in enumerate(PHENOTYPE_ORDER):
        group_x = left + 35 + index * group_width
        actual_height = chart_height * actual_counts[index] / max_count
        theoretical_height = chart_height * theoretical_counts[index] / max_count
        actual_x = group_x
        theoretical_x = group_x + bar_width + 8
        actual_y = bottom - actual_height
        theoretical_y = bottom - theoretical_height

        parts.append(f'<rect x="{actual_x}" y="{actual_y}" width="{bar_width}" height="{actual_height}" fill="#3b82f6" rx="3"/>')
        parts.append(f'<rect x="{theoretical_x}" y="{theoretical_y}" width="{bar_width}" height="{theoretical_height}" fill="#f59e0b" rx="3"/>')
        parts.append(f'<text x="{actual_x + bar_width / 2}" y="{actual_y - 6}" text-anchor="middle" font-size="12">{actual_counts[index]}</text>')
        parts.append(f'<text x="{theoretical_x + bar_width / 2}" y="{theoretical_y - 6}" text-anchor="middle" font-size="12">{theoretical_counts[index]:.1f}</text>')
        parts.append(f'<text x="{group_x + bar_width}" y="{bottom + 24}" text-anchor="middle" font-size="13">{escape(phenotype)}</text>')

    parts.append('</svg>')
    html = ''.join(parts)
    if HTML is not None:
        display(HTML(html))
    else:
        print("matplotlib 不可用，当前统计为：", dict(zip(PHENOTYPE_ORDER, actual_counts)))


def plot_results(records):
    total = len(records)
    phenotype_counts = Counter(row["性状组合"] for row in records)
    actual_counts = [phenotype_counts[item] for item in PHENOTYPE_ORDER]
    theoretical_counts = [THEORETICAL_PHENOTYPE_RATIO[item] * total for item in PHENOTYPE_ORDER]

    if plt is None:
        plot_results_svg(records)
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    x = range(len(PHENOTYPE_ORDER))
    width = 0.36
    axes[0].bar([i - width / 2 for i in x], actual_counts, width=width, label="实际次数")
    axes[0].bar([i + width / 2 for i in x], theoretical_counts, width=width, label="理论期望次数")
    axes[0].set_title(f"性状组合统计（n={total}）")
    axes[0].set_ylabel("次数")
    axes[0].set_xticks(list(x))
    axes[0].set_xticklabels(PHENOTYPE_ORDER, rotation=20)
    axes[0].legend()

    axes[1].pie(
        actual_counts,
        labels=PHENOTYPE_ORDER,
        autopct="%1.1f%%",
        startangle=90,
    )
    axes[1].set_title("实际性状组合比例")

    plt.tight_layout()
    plt.show()


def run_and_show(n=50, seed=None):
    records = run_random_experiments(n=n, seed=seed)
    phenotype_rows, genotype_rows = summarize_results(records)

    print("前 10 次实验记录：")
    show_table(records[:10])

    print("性状组合汇总：")
    show_table(phenotype_rows)

    print("基因型汇总：")
    show_table(genotype_rows)

    plot_results(records)
    return records

## 1. 查看亲本可产生的配子

`WwYy` 亲本理论上可以产生 `WY`、`Wy`、`wY`、`wy` 四种配子，比例为 `1:1:1:1`。

In [3]:
gamete_rows = [
    {"配子": gamete, "理论概率": probability}
    for gamete, probability in possible_gametes(PARENT_GENOTYPE).items()
]
show_table(gamete_rows)

{'配子': 'WY', '理论概率': 0.25}
{'配子': 'Wy', '理论概率': 0.25}
{'配子': 'wY', '理论概率': 0.25}
{'配子': 'wy', '理论概率': 0.25}


## 2. 随机连续实验

修改 `n` 可以改变实验次数。`seed` 用来固定随机结果，方便重复展示；如果想每次都得到不同结果，可以把 `seed=None`。

In [12]:
records = run_and_show(n=100, seed=32)

前 10 次实验记录：
{'实验序号': 1, '亲本甲配子': 'WY', '亲本乙配子': 'Wy', '胚乳性状': '非糯性', '粒形': '长粒', '后代基因型': 'WWYy', '性状组合': '非糯性长粒'}
{'实验序号': 2, '亲本甲配子': 'Wy', '亲本乙配子': 'WY', '胚乳性状': '非糯性', '粒形': '长粒', '后代基因型': 'WWYy', '性状组合': '非糯性长粒'}
{'实验序号': 3, '亲本甲配子': 'Wy', '亲本乙配子': 'wY', '胚乳性状': '非糯性', '粒形': '长粒', '后代基因型': 'WwYy', '性状组合': '非糯性长粒'}
{'实验序号': 4, '亲本甲配子': 'wy', '亲本乙配子': 'WY', '胚乳性状': '非糯性', '粒形': '长粒', '后代基因型': 'WwYy', '性状组合': '非糯性长粒'}
{'实验序号': 5, '亲本甲配子': 'Wy', '亲本乙配子': 'Wy', '胚乳性状': '非糯性', '粒形': '短粒', '后代基因型': 'WWyy', '性状组合': '非糯性短粒'}
{'实验序号': 6, '亲本甲配子': 'Wy', '亲本乙配子': 'Wy', '胚乳性状': '非糯性', '粒形': '短粒', '后代基因型': 'WWyy', '性状组合': '非糯性短粒'}
{'实验序号': 7, '亲本甲配子': 'WY', '亲本乙配子': 'WY', '胚乳性状': '非糯性', '粒形': '长粒', '后代基因型': 'WWYY', '性状组合': '非糯性长粒'}
{'实验序号': 8, '亲本甲配子': 'wY', '亲本乙配子': 'wY', '胚乳性状': '糯性', '粒形': '长粒', '后代基因型': 'wwYY', '性状组合': '糯性长粒'}
{'实验序号': 9, '亲本甲配子': 'Wy', '亲本乙配子': 'WY', '胚乳性状': '非糯性', '粒形': '长粒', '后代基因型': 'WWYy', '性状组合': '非糯性长粒'}
{'实验序号': 10, '亲本甲配子': 'wy', '亲本乙配子': 'wY', '胚乳性状': '糯性', '粒形': 

## 3. 人工指定两个亲本的配子

在下面的代码中修改 `gamete1` 和 `gamete2`，即可指定一次受精过程。可选配子为：`WY`、`Wy`、`wY`、`wy`。

In [5]:
manual_result = make_offspring_from_gametes(gamete1="WY", gamete2="wy")
show_table([manual_result])

{'亲本甲配子': 'WY', '亲本乙配子': 'wy', '胚乳性状': '非糯性', '粒形': '长粒', '后代基因型': 'WwYy', '性状组合': '非糯性长粒'}


## 4. 多次手动指定配子

如果想展示 16 种受精组合，可以运行下面这段代码。

In [6]:
gametes = list(possible_gametes(PARENT_GENOTYPE).keys())
all_manual_records = []

for gamete1 in gametes:
    for gamete2 in gametes:
        all_manual_records.append(make_offspring_from_gametes(gamete1, gamete2))

show_table(all_manual_records)

phenotype_rows, genotype_rows = summarize_results(all_manual_records)
print("16 种等概率组合的性状统计：")
show_table(phenotype_rows)
plot_results(all_manual_records)

{'亲本甲配子': 'WY', '亲本乙配子': 'WY', '胚乳性状': '非糯性', '粒形': '长粒', '后代基因型': 'WWYY', '性状组合': '非糯性长粒'}
{'亲本甲配子': 'WY', '亲本乙配子': 'Wy', '胚乳性状': '非糯性', '粒形': '长粒', '后代基因型': 'WWYy', '性状组合': '非糯性长粒'}
{'亲本甲配子': 'WY', '亲本乙配子': 'wY', '胚乳性状': '非糯性', '粒形': '长粒', '后代基因型': 'WwYY', '性状组合': '非糯性长粒'}
{'亲本甲配子': 'WY', '亲本乙配子': 'wy', '胚乳性状': '非糯性', '粒形': '长粒', '后代基因型': 'WwYy', '性状组合': '非糯性长粒'}
{'亲本甲配子': 'Wy', '亲本乙配子': 'WY', '胚乳性状': '非糯性', '粒形': '长粒', '后代基因型': 'WWYy', '性状组合': '非糯性长粒'}
{'亲本甲配子': 'Wy', '亲本乙配子': 'Wy', '胚乳性状': '非糯性', '粒形': '短粒', '后代基因型': 'WWyy', '性状组合': '非糯性短粒'}
{'亲本甲配子': 'Wy', '亲本乙配子': 'wY', '胚乳性状': '非糯性', '粒形': '长粒', '后代基因型': 'WwYy', '性状组合': '非糯性长粒'}
{'亲本甲配子': 'Wy', '亲本乙配子': 'wy', '胚乳性状': '非糯性', '粒形': '短粒', '后代基因型': 'Wwyy', '性状组合': '非糯性短粒'}
{'亲本甲配子': 'wY', '亲本乙配子': 'WY', '胚乳性状': '非糯性', '粒形': '长粒', '后代基因型': 'WwYY', '性状组合': '非糯性长粒'}
{'亲本甲配子': 'wY', '亲本乙配子': 'Wy', '胚乳性状': '非糯性', '粒形': '长粒', '后代基因型': 'WwYy', '性状组合': '非糯性长粒'}
{'亲本甲配子': 'wY', '亲本乙配子': 'wY', '胚乳性状': '糯性', '粒形': '长粒', '后代基因型': 'wwYY', '性状组合'

In [ ]:
import csv
import os


def make_results_svg(records):
    total = len(records)
    phenotype_counts = Counter(row["性状组合"] for row in records)
    actual_counts = [phenotype_counts[item] for item in PHENOTYPE_ORDER]
    theoretical_counts = [THEORETICAL_PHENOTYPE_RATIO[item] * total for item in PHENOTYPE_ORDER]
    max_count = max(actual_counts + theoretical_counts + [1])

    width = 860
    height = 430
    left = 70
    bottom = 330
    chart_height = 240
    group_width = 170
    bar_width = 34

    parts = [
        f'<svg width="{width}" height="{height}" viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{width / 2}" y="32" text-anchor="middle" font-size="20" font-weight="700">性状组合统计（n={total}）</text>',
        f'<line x1="{left}" y1="{bottom}" x2="{width - 60}" y2="{bottom}" stroke="#222"/>',
        f'<line x1="{left}" y1="{bottom}" x2="{left}" y2="{bottom - chart_height}" stroke="#222"/>',
        '<rect x="640" y="56" width="18" height="18" fill="#3b82f6"/>',
        '<text x="666" y="70" font-size="14">实际次数</text>',
        '<rect x="740" y="56" width="18" height="18" fill="#f59e0b"/>',
        '<text x="766" y="70" font-size="14">理论期望次数</text>',
    ]

    for tick in range(0, 5):
        value = max_count * tick / 4
        y = bottom - chart_height * tick / 4
        parts.append(f'<line x1="{left - 4}" y1="{y}" x2="{width - 60}" y2="{y}" stroke="#e5e7eb"/>')
        parts.append(f'<text x="{left - 10}" y="{y + 5}" text-anchor="end" font-size="12">{value:.0f}</text>')

    for index, phenotype in enumerate(PHENOTYPE_ORDER):
        group_x = left + 35 + index * group_width
        actual_height = chart_height * actual_counts[index] / max_count
        theoretical_height = chart_height * theoretical_counts[index] / max_count
        actual_x = group_x
        theoretical_x = group_x + bar_width + 8
        actual_y = bottom - actual_height
        theoretical_y = bottom - theoretical_height

        parts.append(f'<rect x="{actual_x}" y="{actual_y}" width="{bar_width}" height="{actual_height}" fill="#3b82f6" rx="3"/>')
        parts.append(f'<rect x="{theoretical_x}" y="{theoretical_y}" width="{bar_width}" height="{theoretical_height}" fill="#f59e0b" rx="3"/>')
        parts.append(f'<text x="{actual_x + bar_width / 2}" y="{actual_y - 6}" text-anchor="middle" font-size="12">{actual_counts[index]}</text>')
        parts.append(f'<text x="{theoretical_x + bar_width / 2}" y="{theoretical_y - 6}" text-anchor="middle" font-size="12">{theoretical_counts[index]:.1f}</text>')
        parts.append(f'<text x="{group_x + bar_width}" y="{bottom + 24}" text-anchor="middle" font-size="13">{escape(phenotype)}</text>')

    parts.append('</svg>')
    return ''.join(parts)


def save_records_csv(records, path):
    fieldnames = ["实验序号", "亲本甲配子", "亲本乙配子", "后代基因型", "胚乳性状", "粒形", "性状组合"]
    with open(path, "w", encoding="utf-8-sig", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows({field: row.get(field, "") for field in fieldnames} for row in records)


def save_summary_csv(records, path):
    phenotype_rows, _ = summarize_results(records)
    fieldnames = ["性状组合", "数量", "实际比例", "理论比例"]
    with open(path, "w", encoding="utf-8-sig", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(phenotype_rows)


def save_trait_stats_image(records, path):
    svg = make_results_svg(records)
    with open(path, "w", encoding="utf-8") as file:
        file.write(svg)


def save_experiment_files(records, output_dir="outputs"):
    os.makedirs(output_dir, exist_ok=True)
    n = len(records)
    image_path = os.path.join(output_dir, f"trait_combination_stats_n{n}.svg")
    records_csv_path = os.path.join(output_dir, f"experiment_results_n{n}.csv")
    summary_csv_path = os.path.join(output_dir, f"phenotype_summary_n{n}.csv")

    save_trait_stats_image(records, image_path)
    save_records_csv(records, records_csv_path)
    save_summary_csv(records, summary_csv_path)

    return {
        "性状组合统计图片": image_path,
        "实验结果CSV": records_csv_path,
        "性状组合汇总CSV": summary_csv_path,
    }


saved_files = save_experiment_files(records)
saved_files
